In [1]:
# aim: sleep staging of icu patients' data. first step is to create hdf5 file.

In [2]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/ICU-Sleep')
from airgo_features import airgo_actigraphy_features

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file

sys.path.append('/home/wolfgang/repos/AirGoSleepPyT0')
from segment_signal import *
from collections import Counter
from scipy import io as sio

import torch as th
th.cuda.empty_cache()
from utils import softmax
from braindecode.torch_ext.util import np_to_var, var_to_np

from tqdm import tqdm

from sleep_staging_functions import *

In [3]:
def clip_z_normalize(signal):

    signal_clipped = np.clip(signal.dropna(), np.percentile(signal.dropna(),1), np.percentile(signal.dropna(),99))
    mean = np.mean(signal_clipped.astype(np.float32))
    std = np.std(signal_clipped.astype(np.float32))
    signal = (signal - mean)/std

    return signal

 
def myprint(seg_mask):
    sm = Counter([x.split('_')[0] for x in seg_mask])
    for ex in seg_mask_explanation:
        if ex in sm:
            print(('%s: %d/%d, %g%%'%(ex,sm[ex],len(seg_mask),sm[ex]*100./len(seg_mask))))
    

In [5]:

seg_mask_explanation = [
    'normal',
    'around sleep stage change point',
    'NaN in sleep stage',
    'NaN in signal',
    'overly high/low amplitude',
    'flat signal',
    'NaN in feature',
    'NaN in spectrum',
    'muscle artifact',
    'spurious spectrum',
    'missing or additional R peak']


In [6]:
# first time, use just 10Hz version, then append to same data, i.e. give each sleep staging 
# result a name/id and if more sleep stage models are applied, just more arrays with different ids are appended.


In [7]:
# input_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data'
input_dir = '/media/ssd2/icu_sleepstaging'
output_dir = '/media/ssd2/icu_sleepstaging'

verbose = False
files = os.listdir(input_dir)
files.sort()
print(files[:3])

window_time = 270
step_time =30
start_end_remove_window_num=1
n_jobs=-1
to_remove_mean=False


['icusleep_001.h5', 'icusleep_002.h5', 'icusleep_003.h5']


In [8]:

# add sleep stage prediction #1, 06/22/2020:
# model_id = 'amendsumeffort'
# input_signal = 'band_smooth'
# models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models'

# add sleep stage prediction #2, 06/24/2020:
model_id = 'activity1sec'
input_signal = 'acc_ai_1sec'
models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models'

# cnn_model = os.path.join(models_dir, 'CNN_AirGoFilter_AmendSumEffort.pth')
cnn_model = os.path.join(models_dir, 'CNN_ActivityIndex1sec.pth')
cnn_model = th.load(cnn_model);
cnn_model.eval();

# lstm_model = os.path.join(models_dir, 'RNN_AirGoFilter_AmendSumEffort.pth')
lstm_model = os.path.join(models_dir, 'RNN_ActivityIndex1sec.pth')
lstm_model = th.load(lstm_model);
lstm_model.eval();


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/torch/serialization.py:657: SourceChangeWarning: source code of class 'mymodel.CHESTSleepNet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)


In [9]:
def print_data_availability(series):
    
        print(f'shape total: {series.shape}')
        print(f'shape w/o NA: {series.dropna().shape}')
        print(f'# NA: {pd.isna(series).sum()}')

        print('in hours:')
        print(f'shape total: {series.shape[0]/fs/3600 :.1f}')
        print(f'shape w/o NA: {series.dropna().shape[0]/fs/3600:.1f}')
        print(f'# NA: {pd.isna(series).sum()/fs/3600:.1f}')


In [10]:
def cnn_and_lstm_predict(X, cnn_model, lstm_model):

    with th.no_grad():
        output, H = cnn_model(X)
        output = var_to_np(output)
        H = var_to_np(H)

        H = np.expand_dims(H, axis=0)
        H = np_to_var(H)
        H = H.cuda()

        yp = lstm_model(H)[0]
        yp = var_to_np(yp)
        yp = np.squeeze(yp)

        yp = softmax(yp)

    th.cuda.empty_cache()
    
    del H, cnn_model, lstm_model, output
    
    return yp


In [11]:
len(files)

133

In [12]:
for file_tmp in tqdm(files[:60]):
    try:
#     if 1:
        
#         if os.path.exists(os.path.join(output_dir, file_tmp)):
#             continue
        

        data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
        data[np.isinf(data)] = np.nan
        data = data.astype('float64')         
        fs = hdr['fs']
        data.columns = [x.lower() for x in data.columns]

        hdr['study_id'] = np.int32(hdr['study_id'])
        hdr['MRN'] = np.int32(hdr['MRN'])
        hdr['fs'] = np.int32(hdr['fs'])
        hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

        if f'stage_pred_{model_id}' in data.columns:
            print(f'stage_pred_{model_id} is already contained in data. Skip.')
            continue
        
        if not 'band' in data.columns:
            print('No AirGo data at all.')
            continue
            
        # print(data.band.dropna().shape[0])
        if data.band.dropna().shape[0] == 0:
            print('No AirGo data, seems like wrong category assigned')
        #     continue # todo
            continue
            
        if data.acc_ai_10sec.dropna().mean() > 1:
            print(file_tmp)
            print('data.acc_ai_10sec.dropna().mean() > 1 --> V4.')
            continue
            
        if any(data.band.dropna()>5000):
            print(file_tmp)
            print('any(pd.Series(signal[0,:]).dropna()>5000 --> V4.')
            continue
                
        data.rename({'spo2%':'spo2'}, axis=1, inplace=True)

#         if not 'acc_ai_1sec' in data.columns:
            
#             features_to_keep = ['art1d', 'art1s', 'hr', 'spo2', 
#                             'accx', 'accy', 'accz', 'band']
#             features_to_keep = [x for x in features_to_keep if x in data.columns]
#             data = data[features_to_keep]


#             # also compute actigraphy activity:
#             data = airgo_actigraphy_features(data, fs=10)
#             data.drop(['accx', 'accy', 'accz'], axis=1, inplace=True)

        data['band_smooth'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True, min_periods=1).mean().values

        if verbose:
            print_data_availability(data.band_smooth)

        if input_signal == 'band_smooth':
            signal = data.band_smooth.values
            signal = signal[np.newaxis, :]

            amplitude_thres = 4075
            amplitude_thres_low = 400 # only in use for AirGo.
            std_thres1=0.00001
            std_thres2=0.00001

            if any(pd.Series(signal[0,:]).dropna()>5000):
                print('airgo version pre V5, remove amplitude thresholds')
                amplitude_thres = 100000000
                amplitude_thres_low = 0 # only in use for AirGo.

        elif input_signal == 'acc_ai_1sec':
            signal = data.band_smooth.values
            signal = signal[np.newaxis, :]
            amplitude_thres = None
            amplitude_thres_low = None
            std_thres1=0
            std_thres2=0
                
        if any(pd.Series(signal[0,:]).dropna()>5000):
            print('airgo version pre V5, remove amplitude thresholds')
            amplitude_thres = 100000000
            amplitude_thres_low = 0 # only in use for AirGo.

        segs, seg_mask, seg_start_pos = segment_signal_only(signal, window_time, step_time, fs, 
                                start_end_remove_window_num=start_end_remove_window_num, 
                                amplitude_thres=amplitude_thres, amplitude_thres_low=amplitude_thres_low,
                            n_jobs=n_jobs, to_remove_mean=to_remove_mean)

        normal_only = True
        if normal_only:
            good_ids = np.where(np.in1d(seg_mask,seg_mask_explanation[:2]))[0]
            if len(good_ids)<=0:
                myprint(seg_mask)
                raise ValueError('No normal segments')
            segs = segs[good_ids]
            seg_start_pos = seg_start_pos[good_ids]
        else:
            good_ids = np.arange(len(seg_mask))

        # REMOVE INF and NAN if still here...
        # segs.shape
        isgood = np.isfinite(segs).all(axis=2).flatten()
        segs = segs[isgood,:,:]
        seg_start_pos = seg_start_pos[isgood]

        X = th.tensor(segs).float()

        yp = cnn_and_lstm_predict(X, cnn_model, lstm_model)

        yp = np.argmax(yp, axis=1) + 1

        data[f'stage_pred_{model_id}'] = np.nan
        stage_loc = data.iloc[seg_start_pos].index
        data.loc[stage_loc, f'stage_pred_{model_id}'] = yp
        data[f'stage_pred_{model_id}'].fillna(method='pad',
                                              limit=30*fs-1, inplace=True)
        write_to_hdf5_file(data, os.path.join(output_dir, file_tmp), hdr=hdr, overwrite=True)

        th.cuda.empty_cache()

        
    except Exception as e:
        print('Exception for:' + file_tmp)
        print(e)


  2%|▏         | 1/60 [00:07<07:39,  7.79s/it]

icusleep_001.h5
any(pd.Series(signal[0,:]).dropna()>5000 --> V4.


  3%|▎         | 2/60 [00:10<06:10,  6.39s/it]

icusleep_002.h5
data.acc_ai_10sec.dropna().mean() > 1 --> V4.


  5%|▌         | 3/60 [00:18<06:26,  6.79s/it]

...

 28%|██▊       | 17/60 [03:52<07:32, 10.52s/it]

icusleep_030.h5
data.acc_ai_10sec.dropna().mean() > 1 --> V4.


 38%|███▊      | 23/60 [06:08<11:14, 18.24s/it]

icusleep_038.h5
any(pd.Series(signal[0,:]).dropna()>5000 --> V4.


100%|██████████| 60/60 [20:15<00:00, 20.26s/it]


In [33]:
plt.figure()
plt.plot(data.stage_pred_amendsumeffort)
plt.plot(data.stage_pred_activity1sec)
plt.legend(['SS Respiration', 'SS Actigraphy'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [30]:
file_tmp = files[0]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())
print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_001.h5
11816.030061918515
0.650245164558424
1.7812793742598172
7.750581188501006
18.238370592566636


In [29]:
file_tmp = files[1]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())
print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_002.h5
14408.662149802583
1.6624210968621398
2.5766504691943943
1.6578910923083754
3.3506949669361767


In [28]:
file_tmp = files[2]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())
print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_003.h5
11322.106125364244
2.7530216469906343
2.9203390120901425
15.179172718425525
1.723460052557705


In [27]:
file_tmp = files[45]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())
print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_077.h5
2052.1389146923557
0.017738386312119013
0.029311174013175473
-0.12187736501419301
-0.5680418682122933


In [26]:
file_tmp = files[46]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())
print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_078.h5
2269.5901871324263
0.02374482969413707
0.042456276884126724
0.05986790195457662
-0.8670821509434555


In [25]:
file_tmp = files[47]
print(file_tmp)
data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
data[np.isinf(data)] = np.nan
data = data.astype('float64')         
fs = hdr['fs']
data.columns = [x.lower() for x in data.columns]
print(data.band.dropna().mean())
print(data.acc_ai_10sec.dropna().mean())
print(data.acc_ai_10sec.dropna().std())

print(data.accy_1sec.dropna().mean())
print(data.accz_1sec.dropna().mean())

icusleep_079.h5
2053.6780576809206
0.015587992736577773
0.024872497270584187
-0.10858218791589684
-0.8463059563051827


-0.8463059563051827

In [20]:
data.stage_pred_amendsumeffort.dropna()

2019-04-11 18:54:29.000    5.0
2019-04-11 18:54:29.100    5.0
2019-04-11 18:54:29.200    5.0
2019-04-11 18:54:29.300    5.0
2019-04-11 18:54:29.400    5.0
                          ... 
2019-04-13 10:59:58.500    3.0
2019-04-13 10:59:58.600    3.0
2019-04-13 10:59:58.700    3.0
2019-04-13 10:59:58.800    3.0
2019-04-13 10:59:58.900    3.0
Name: stage_pred_amendsumeffort, Length: 1346100, dtype: float64

In [21]:
data.columns

Index(['acc_ai_10sec', 'acc_ai_1sec', 'acc_enmo', 'acc_enmo_10sec',
       'acc_enmo_1sec', 'accx_1sec', 'accy_1sec', 'accz_1sec', 'art1d',
       'art1s', 'band', 'band_smooth', 'hr', 'position_cluster', 'spo2',
       'stage_pred_amendsumeffort'],
      dtype='object')